# Residual Stream Attribution

Trace how each component builds the final residual: per-position attribution,
directional decomposition, component overlap, cumulative buildup, and
logit attribution by component.

## Why This Matters

Residual stream stream attribution characterizes the shared information bus that all transformer components read from and write to. Because the residual stream carries all inter-component communication, understanding its structure is fundamental to understanding the model as a whole.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis
- [nostalgebraist (2020) "interpreting GPT: the logit lens"](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens) — Project intermediate residuals through the unembedding

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.residual_stream_attribution import (
    per_position_attribution, directional_attribution,
    component_overlap_matrix, cumulative_buildup,
    logit_attribution_by_component,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Per-Position Attribution

Which component contributes the most to the residual at the last position?

In [ ]:
result = per_position_attribution(model, tokens, position=-1)
print(f"Total residual norm: {result['total_residual_norm']:.4f}")
print(f"Largest contributor: {result['largest_contributor']['name']}\n")
for c in result['per_component']:
    bar = '█' * int(c['norm'] * 50)
    print(f"  {c['name']:10s}: norm={c['norm']:.4f} {bar}")

## Directional Attribution

How much does each component contribute along a specific direction?

In [ ]:
direction = jax.random.normal(jax.random.PRNGKey(0), (32,))
result = directional_attribution(model, tokens, direction)
print(f"Total projection: {result['total_projection']:.4f}")
print(f"Sum of components: {result['sum_projections']:.4f}\n")
for c in result['per_component']:
    sign = '+' if c['projection'] > 0 else '-'
    print(f"  {c['name']:10s}: {sign}{c['abs_projection']:.4f} (raw: {c['projection']:+.4f})")

## Component Overlap Matrix

How much do different components' contributions overlap?

In [ ]:
result = component_overlap_matrix(model, tokens)
print(f"Mean absolute overlap: {result['mean_overlap']:.4f}\n")
print('Overlap matrix (cosine):')
header = '          ' + '  '.join(f'{n:>8s}' for n in result['names'])
print(header)
for i, name in enumerate(result['names']):
    row = f'{name:10s}' + '  '.join(f'{float(result["overlap_matrix"][i,j]):+.3f}' for j in range(result['n_components']))
    print(row)

## Cumulative Buildup

How does the residual evolve through layers?

In [ ]:
result = cumulative_buildup(model, tokens)
print(f"Norm growth rate: {result['norm_growth_rate']:.4f}")
print(f"Final prediction: token {result['final_prediction']}\n")
for s in result['stages']:
    print(f"  Layer {s['layer']}: norm={s['residual_norm']:.4f}, "
          f"pred={s['top_prediction']}, conf={s['confidence']:.4f}")

## Logit Attribution by Component

Which components contribute most to the logit of the predicted token?

In [ ]:
result = logit_attribution_by_component(model, tokens)
print(f"Target token: {result['target_token']}")
print(f"Total logit: {result['total_logit']:.4f}\n")
for c in result['per_component']:
    sign = '+' if c['logit_contribution'] > 0 else '-'
    print(f"  {c['name']:10s}: {sign}{c['abs_contribution']:.4f}")